# Parcel Box Detector - YOLOv8 Training (Colab)

Pipeline stages covered here: YOLOv8 파인튜닝 -> 정확도 평가 -> ONNX 변환.

Before running: upload `parcel_yolo_dataset.zip` (produced by `isaac_sim/scripts/convert_to_yolo.py`)
to this Colab session, either via the Files pane or Google Drive.

In [ ]:
!pip install -q ultralytics

## 1. Get the dataset onto the Colab VM

Option A: upload the zip directly (Files pane, left sidebar) then run the unzip cell below.

Option B: mount Google Drive if the zip lives there instead.

In [ ]:
# Option A: local upload
from google.colab import files
uploaded = files.upload()  # select parcel_yolo_dataset.zip
zip_name = list(uploaded.keys())[0]

In [ ]:
# Option B: Google Drive instead of the upload above
# from google.colab import drive
# drive.mount('/content/drive')
# zip_name = '/content/drive/MyDrive/parcel_yolo_dataset.zip'

In [ ]:
import zipfile

DATASET_DIR = "/content/parcel_yolo"
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(DATASET_DIR)
print("Extracted to", DATASET_DIR)

In [ ]:
# Rewrite data.yaml's 'path' to point at this Colab session's copy
# (the original file was written with the path from wherever it was generated on the local PC).
import glob
import os

data_yaml_path = glob.glob(f"{DATASET_DIR}/**/data.yaml", recursive=True)[0]
dataset_root = os.path.dirname(data_yaml_path)

lines = open(data_yaml_path).read().splitlines()
lines = [f"path: {dataset_root}" if line.startswith("path:") else line for line in lines]
open(data_yaml_path, "w").write("\n".join(lines) + "\n")
print(open(data_yaml_path).read())

## 2. Fine-tune YOLOv8

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # nano: fastest to fine-tune; swap for yolov8s.pt if accuracy is more important than speed
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    project="parcel_box_runs",
    name="yolov8n_parcel",
)

## 3. Evaluate accuracy (mAP / Precision / Recall)

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 4. Export to ONNX (for the real-time inference step later)

In [ ]:
onnx_path = model.export(format="onnx", opset=12, simplify=True)
print("Exported:", onnx_path)

## 5. Download the trained weights + ONNX back to your PC

In [ ]:
from google.colab import files

best_pt = "parcel_box_runs/yolov8n_parcel/weights/best.pt"
files.download(best_pt)
files.download(onnx_path)